# Stage 2: Temporal Deforestation Detection -- GPU Training on Google Colab

**Project:** AI-Based Forest Cover Segmentation and Temporal Deforestation Detection Using U-Net

**What this notebook does:**
1. Checks GPU availability
2. Clones the project from GitHub (`testing` branch)
3. Writes updated Stage 2 v2 source files into `src/`
4. Accepts the Forest-Change dataset via file upload or Google Drive
5. Verifies dataset folder structure
6. Runs **Experiments A, B, C** sequentially
7. Selects the best experiment by **validation IoU only** (test set is never touched)
8. Sweeps thresholds [0.2, 0.3, 0.4, 0.5, 0.6, 0.7] on **validation** set
9. Evaluates the **test set once** at the best validation threshold
10. Downloads `best_temporal_model.pth`

> **Before running:** Enable GPU.  
> **Runtime** > **Change runtime type** > **T4 GPU** > **Save** > **Reconnect**

## 1. Check GPU

In [ ]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')


## 2. Clone the project from GitHub (testing branch)

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/vikaskmalipatil/Deforestation-Detection-from-Satellite-Images.git'
BRANCH   = 'testing'
PROJ_DIR = '/content/deforestation'
if not os.path.exists(PROJ_DIR):
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,PROJ_DIR], check=True)
    print(f'Cloned {BRANCH} branch to {PROJ_DIR}')
else:
    print('Project directory already exists, skipping clone.')
os.chdir(PROJ_DIR)
print(f'Working directory: {os.getcwd()}')


## 3. Install dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '--quiet', 'Pillow', 'matplotlib'], check=True)
import torch, PIL, matplotlib, numpy
print(f'torch {torch.__version__} | Pillow {PIL.__version__} | matplotlib {matplotlib.__version__} | numpy {numpy.__version__}')


## 4. Write updated Stage 2 source files (v2)

The GitHub `testing` branch contains the original Stage 2 files.  
These cells overwrite them with the **v2 improvements**:
- `diff_channels` flag for 9-channel `[A, B, |A-B|]` input
- `compute_pos_weight()` from full training set (exact, not estimated)
- `FocalLoss`, `FocalDiceLoss`, `WeightedBCEDiceLoss`
- `bottleneck_dropout` in `TemporalUNet`

In [ ]:
from pathlib import Path
import sys

dataset_py = 'import random\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom PIL import Image\n\nIMAGE_SIZE = 256\nTARGET_SIZE = (IMAGE_SIZE, IMAGE_SIZE)\nSEED = 42\n\n\nclass TemporalForestDataset(Dataset):\n    """\n    Temporal bi-image dataset.\n    Labels: mask > 0  =>  deforestation (multi-value grayscale intensity encoding).\n    diff_channels=True  =>  9-ch input [A_RGB, B_RGB, |A-B|_RGB]\n    diff_channels=False =>  6-ch input [A_RGB, B_RGB]\n    """\n    def __init__(self, split_dir, augment=False, target_size=TARGET_SIZE, diff_channels=False):\n        self.split_dir = Path(split_dir)\n        self.augment = augment\n        self.target_size = target_size\n        self.diff_channels = diff_channels\n        self.a_dir     = self.split_dir / "A"\n        self.b_dir     = self.split_dir / "B"\n        self.label_dir = self.split_dir / "label"\n        if not (self.a_dir.exists() and self.b_dir.exists() and self.label_dir.exists()):\n            raise FileNotFoundError(f"Missing A/B/label dirs in {split_dir}")\n        self.image_names = sorted(f.name for f in self.a_dir.glob("*.png"))\n        if not self.image_names:\n            raise RuntimeError(f"No PNG files in {self.a_dir}")\n\n    def __len__(self):\n        return len(self.image_names)\n\n    def _augment(self, a, b, m):\n        if random.random() > 0.5:\n            a, b, m = np.fliplr(a), np.fliplr(b), np.fliplr(m)\n        if random.random() > 0.5:\n            a, b, m = np.flipud(a), np.flipud(b), np.flipud(m)\n        k = random.choice([0, 1, 2, 3])\n        if k:\n            a = np.rot90(a, k, (0, 1)); b = np.rot90(b, k, (0, 1)); m = np.rot90(m, k, (0, 1))\n        return a.copy(), b.copy(), m.copy()\n\n    def __getitem__(self, idx):\n        n = self.image_names[idx]\n        def _load(p):\n            img = Image.open(p).convert("RGB")\n            if img.size != self.target_size:\n                img = img.resize(self.target_size, Image.Resampling.BILINEAR)\n            return np.array(img, dtype=np.float32)\n        a = _load(self.a_dir / n)\n        b = _load(self.b_dir / n)\n        lbl = Image.open(self.label_dir / n).convert("L")\n        if lbl.size != self.target_size:\n            lbl = lbl.resize(self.target_size, Image.Resampling.NEAREST)\n        m = (np.array(lbl) > 0).astype(np.float32)\n        if self.augment:\n            a, b, m = self._augment(a, b, m)\n        a, b = a / 255.0, b / 255.0\n        at = torch.tensor(a, dtype=torch.float32).permute(2, 0, 1)\n        bt = torch.tensor(b, dtype=torch.float32).permute(2, 0, 1)\n        mt = torch.tensor(m, dtype=torch.float32).unsqueeze(0)\n        if self.diff_channels:\n            img = torch.cat([at, bt, torch.abs(at - bt)], dim=0)\n        else:\n            img = torch.cat([at, bt], dim=0)\n        return img, mt, n\n\n\ndef compute_pos_weight(dataset_root, image_size=IMAGE_SIZE):\n    """Exact neg/pos ratio from full training set. Call once before training."""\n    root = Path(dataset_root)\n    ds = TemporalForestDataset(root / "train", augment=False, target_size=(image_size, image_size))\n    pos, neg = 0, 0\n    for _, m, _ in ds:\n        pos += m.sum().item()\n        neg += (m == 0).sum().item()\n    return float(min(max(neg / (pos + 1e-8), 1.0), 50.0))\n\n\ndef get_temporal_dataloaders(dataset_root, batch_size=8, num_workers=2,\n                              image_size=IMAGE_SIZE, diff_channels=False):\n    root = Path(dataset_root)\n    kw = dict(target_size=(image_size, image_size), diff_channels=diff_channels)\n    tr = TemporalForestDataset(root / "train", augment=True,  **kw)\n    vl = TemporalForestDataset(root / "val",   augment=False, **kw)\n    te = TemporalForestDataset(root / "test",  augment=False, **kw)\n    lkw = dict(num_workers=num_workers, pin_memory=True)\n    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  **lkw),\n            DataLoader(vl, batch_size=batch_size, shuffle=False, **lkw),\n            DataLoader(te, batch_size=batch_size, shuffle=False, **lkw))\n'
model_py   = 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nTEMPORAL_CHECKPOINT_PATH = "models/best_temporal_model.pth"\n\n\nclass DoubleConv(nn.Module):\n    def __init__(self, ic, oc, drop=0.0):\n        super().__init__()\n        layers = [\n            nn.Conv2d(ic, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),\n            nn.Conv2d(oc, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),\n        ]\n        if drop > 0:\n            layers.append(nn.Dropout2d(p=drop))\n        self.block = nn.Sequential(*layers)\n\n    def forward(self, x):\n        return self.block(x)\n\n\nclass DiceLoss(nn.Module):\n    def __init__(self, smooth=1e-6):\n        super().__init__()\n        self.smooth = smooth\n\n    def forward(self, logits, targets):\n        p = torch.sigmoid(logits).view(-1)\n        t = targets.view(-1).float()\n        inter = (p * t).sum()\n        return 1.0 - (2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)\n\n\nclass FocalLoss(nn.Module):\n    def __init__(self, gamma=2.0, alpha=0.75):\n        super().__init__()\n        self.gamma = gamma\n        self.alpha = alpha\n\n    def forward(self, logits, targets):\n        t = targets.float()\n        bce = F.binary_cross_entropy_with_logits(logits, t, reduction="none")\n        p = torch.sigmoid(logits)\n        p_t = p * t + (1 - p) * (1 - t)\n        w = (1 - p_t) ** self.gamma\n        if self.alpha is not None:\n            a_t = self.alpha * t + (1 - self.alpha) * (1 - t)\n            w = a_t * w\n        return (w * bce).mean()\n\n\nclass WeightedBCEDiceLoss(nn.Module):\n    def __init__(self, pos_weight_val=20.0, bce_w=0.5, dice_w=0.5):\n        super().__init__()\n        self.bce_w  = bce_w\n        self.dice_w = dice_w\n        self.bce  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val]))\n        self.dice = DiceLoss()\n\n    def forward(self, logits, targets):\n        targets = targets.float()\n        self.bce.pos_weight = self.bce.pos_weight.to(logits.device)\n        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)\n\n\nclass FocalDiceLoss(nn.Module):\n    def __init__(self, focal_w=0.5, dice_w=0.5, gamma=2.0, alpha=0.75):\n        super().__init__()\n        self.focal_w = focal_w\n        self.dice_w  = dice_w\n        self.focal   = FocalLoss(gamma=gamma, alpha=alpha)\n        self.dice    = DiceLoss()\n\n    def forward(self, logits, targets):\n        targets = targets.float()\n        return self.focal_w * self.focal(logits, targets) + self.dice_w * self.dice(logits, targets)\n\n\nclass TemporalUNet(nn.Module):\n    """\n    Temporal U-Net for binary deforestation change detection.\n    in_channels=6 : [A_RGB, B_RGB]            (Exp A)\n    in_channels=9 : [A_RGB, B_RGB, |A-B|_RGB] (Exp B, C)\n    """\n    def __init__(self, in_channels=6, out_channels=1, init_features=32, bottleneck_dropout=0.0):\n        super().__init__()\n        f = init_features\n        self.inc   = DoubleConv(in_channels, f)\n        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f,    f*2))\n        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*2,  f*4))\n        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*4,  f*8))\n        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*8,  f*16, drop=bottleneck_dropout))\n        self.up1 = nn.ConvTranspose2d(f*16, f*8, 2, stride=2); self.cu1 = DoubleConv(f*16, f*8)\n        self.up2 = nn.ConvTranspose2d(f*8,  f*4, 2, stride=2); self.cu2 = DoubleConv(f*8,  f*4)\n        self.up3 = nn.ConvTranspose2d(f*4,  f*2, 2, stride=2); self.cu3 = DoubleConv(f*4,  f*2)\n        self.up4 = nn.ConvTranspose2d(f*2,  f,   2, stride=2); self.cu4 = DoubleConv(f*2,  f)\n        self.outc = nn.Conv2d(f, out_channels, 1)\n\n    def forward(self, x):\n        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2)\n        x4 = self.down3(x3); x5 = self.down4(x4)\n        u = self.cu1(torch.cat([x4, self.up1(x5)], 1))\n        u = self.cu2(torch.cat([x3, self.up2(u)],  1))\n        u = self.cu3(torch.cat([x2, self.up3(u)],  1))\n        u = self.cu4(torch.cat([x1, self.up4(u)],  1))\n        return self.outc(u)\n'

Path('src').mkdir(exist_ok=True)
Path('src/temporal_dataset.py').write_text(dataset_py, encoding='utf-8')
Path('src/temporal_model.py').write_text(model_py,     encoding='utf-8')
print('Written: src/temporal_dataset.py')
print('Written: src/temporal_model.py')

# Add src to path
sys.path.insert(0, str(Path('src').resolve()))
print('src/ added to sys.path')


## 5. Upload the Forest-Change dataset

**On your Windows machine**, ZIP the extracted dataset folder:
```
  data\Forest-Change\extracted\   <-- ZIP this entire folder
```
The ZIP must produce, when extracted, the path:
```
  Forest-Change-dataset/images/train/A/*.png
  Forest-Change-dataset/images/train/B/*.png
  Forest-Change-dataset/images/train/label/*.png
  Forest-Change-dataset/images/val/...
  Forest-Change-dataset/images/test/...
```

Run the cell below -> click **Choose Files** -> upload your ZIP.

In [ ]:
import zipfile
from pathlib import Path

DATASET_TARGET = Path('data/Forest-Change/extracted')
DATASET_ROOT   = DATASET_TARGET / 'Forest-Change-dataset/images'

if (DATASET_ROOT / 'train/A').exists():
    print('Dataset already present, skipping upload.')
else:
    from google.colab import files
    print('Upload your Forest-Change ZIP file...')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        zp = Path('/content') / fname
        zp.write_bytes(content)
        print(f'Uploaded: {fname} ({len(content)/1e6:.1f} MB)')
        DATASET_TARGET.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(DATASET_TARGET)
        zp.unlink()
        print(f'Extracted to {DATASET_TARGET}')


### Alternative: Mount Google Drive

In [ ]:
# If your dataset is on Google Drive, uncomment and run this block instead:
#
# from google.colab import drive
# import zipfile
# from pathlib import Path
# drive.mount('/content/drive')
# DRIVE_ZIP = '/content/drive/MyDrive/Forest-Change-dataset.zip'  # <- adjust path
# DATASET_TARGET = Path('data/Forest-Change/extracted')
# DATASET_TARGET.mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(DRIVE_ZIP) as zf:
#     zf.extractall(DATASET_TARGET)
# print('Extracted from Drive.')
print('Uncomment the block above if using Google Drive.')


## 6. Verify dataset structure

In [ ]:
from pathlib import Path
DATASET_ROOT = Path('data/Forest-Change/extracted/Forest-Change-dataset/images')
splits = ['train/A','train/B','train/label','val/A','val/B','val/label','test/A','test/B','test/label']
ok = True
for s in splits:
    p = DATASET_ROOT / s
    if p.exists():
        n = len(list(p.glob('*.png')))
        print(f'  OK  {s:<24} ({n} images)')
    else:
        print(f'  MISSING  {s}'); ok = False
if not ok:
    raise RuntimeError('Dataset structure check failed. Fix paths before continuing.')
print('\nDataset structure verified OK.')


## 7. Training configuration

In [ ]:
import sys, os, csv, json, random, shutil
from pathlib import Path
import torch
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('src').resolve()))
from temporal_dataset import get_temporal_dataloaders, compute_pos_weight
from temporal_model import TemporalUNet, WeightedBCEDiceLoss, FocalDiceLoss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED       = 42
BATCH_SIZE = 8      # 8 works on T4 16 GB with 256x256 images
MAX_EPOCHS = 30
PATIENCE   = 8      # early stopping patience
LR         = 1e-3
NUM_WORKERS = 2

DATASET_ROOT = 'data/Forest-Change/extracted/Forest-Change-dataset/images'
EXP_DIR   = Path('outputs/experiments')
MODEL_DIR = Path('models')
EXP_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    'A': dict(diff_channels=False, in_channels=6, init_features=32, bottleneck_dropout=0.0,
              loss='wbce_dice',  description='6-ch [A,B] | WeightedBCE+Dice | lr->RLP'),
    'B': dict(diff_channels=True,  in_channels=9, init_features=32, bottleneck_dropout=0.0,
              loss='wbce_dice',  description='9-ch [A,B,|A-B|] | WeightedBCE+Dice | lr->RLP'),
    'C': dict(diff_channels=True,  in_channels=9, init_features=32, bottleneck_dropout=0.0,
              loss='focal_dice', description='9-ch [A,B,|A-B|] | Focal+Dice | lr->RLP'),
}
print('Config loaded. Experiments:', list(EXPERIMENTS.keys()))


## 8. Compute pos_weight from full training set

In [ ]:
print('Scanning 270 training pairs for class balance...')
pos_weight_val = compute_pos_weight(DATASET_ROOT)
print(f'pos_weight = {pos_weight_val:.2f}  '
      f'(background outnumbers deforestation by ~{pos_weight_val:.0f}x)')


## 9. Define metric helpers and experiment runner

In [ ]:
import torch.optim as optim

def iou_f(tp, fp, fn):
    d = tp + fp + fn
    return tp / d if d > 0 else 1.0

def dice_f(tp, fp, fn):
    d = 2*tp + fp + fn
    return (2*tp) / d if d > 0 else 1.0

def eval_loader(model, loader, crit, thr=0.5):
    model.eval()
    loss_sum = tp = fp = fn = tn = 0
    with torch.no_grad():
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            lg = model(imgs)
            loss_sum += crit(lg, masks).item() * imgs.size(0)
            pr = (torch.sigmoid(lg) > thr).float()
            tp += (pr * masks).sum().item()
            fp += (pr * (1 - masks)).sum().item()
            fn += ((1 - pr) * masks).sum().item()
            tn += ((1 - pr) * (1 - masks)).sum().item()
    n = len(loader.dataset)
    return dict(loss=loss_sum/n, iou=iou_f(tp,fp,fn), dice=dice_f(tp,fp,fn),
                precision=tp/(tp+fp) if (tp+fp)>0 else 1.0,
                recall=tp/(tp+fn) if (tp+fn)>0 else 1.0,
                tp=tp, fp=fp, fn=fn, tn=tn)

def run_experiment(name, cfg, pw):
    random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    diff     = cfg['diff_channels']
    max_ep   = cfg.get('max_epochs', MAX_EPOCHS)
    patience = cfg.get('patience', PATIENCE)
    print('\n' + '='*60)
    print(f'EXPERIMENT {name}: {cfg["description"]}')
    print('='*60)
    tl, vl, _ = get_temporal_dataloaders(DATASET_ROOT, BATCH_SIZE, NUM_WORKERS, diff_channels=diff)
    print(f'Train: {len(tl.dataset)} | Val: {len(vl.dataset)}')
    model = TemporalUNet(in_channels=cfg['in_channels'], out_channels=1,
                         init_features=cfg['init_features'],
                         bottleneck_dropout=cfg['bottleneck_dropout']).to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Parameters: {params:,}')
    if cfg['loss'] == 'wbce_dice':
        crit = WeightedBCEDiceLoss(pos_weight_val=pw)
        print(f'Loss: WeightedBCE+Dice (pos_weight={pw:.2f})')
    else:
        crit = FocalDiceLoss(gamma=2.0, alpha=0.75)
        print('Loss: Focal(gamma=2.0, alpha=0.75) + Dice')
    opt   = optim.Adam(model.parameters(), lr=LR)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=4, min_lr=1e-5)
    od = EXP_DIR / f'exp_{name}'; od.mkdir(parents=True, exist_ok=True)
    ckpt = od / 'best_model.pth'
    hist = {k: [] for k in ['epoch','train_loss','train_iou','val_loss','val_iou','val_dice','lr']}
    best_iou = no_imp = 0
    for ep in range(1, max_ep + 1):
        model.train()
        tl_sum = tp = fp = fn = 0
        for imgs, masks, _ in tl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            lg = model(imgs)
            loss = crit(lg, masks)
            loss.backward(); opt.step()
            tl_sum += loss.item() * imgs.size(0)
            pr = (torch.sigmoid(lg) > 0.5).float()
            tp += (pr * masks).sum().item()
            fp += (pr * (1 - masks)).sum().item()
            fn += ((1 - pr) * masks).sum().item()
        tl_sum /= len(tl.dataset)
        t_iou = iou_f(tp, fp, fn)
        vm = eval_loader(model, vl, crit)
        sched.step(vm['iou'])
        lr_now = opt.param_groups[0]['lr']
        for k, v in zip(hist, [ep, tl_sum, t_iou, vm['loss'], vm['iou'], vm['dice'], lr_now]):
            hist[k].append(v)
        print(f'Ep[{ep:02}/{max_ep}] TLoss={tl_sum:.4f} TIoU={t_iou:.4f} | '
              f'VLoss={vm["loss"]:.4f} VIoU={vm["iou"]:.4f} VDice={vm["dice"]:.4f} | lr={lr_now:.1e}')
        if vm['iou'] > best_iou:
            best_iou = vm['iou']; no_imp = 0
            torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                        'best_val_iou': best_iou, 'config': cfg}, ckpt)
            print(f'  -> Best: Val IoU={best_iou:.4f}')
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  -> Early stop at epoch {ep}'); break
    with open(od / 'history.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(hist.keys())
        for i in range(len(hist['epoch'])): w.writerow([hist[k][i] for k in hist])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(hist['epoch'], hist['train_loss'], label='train')
    axes[0].plot(hist['epoch'], hist['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(hist['epoch'], hist['train_iou'],  label='train')
    axes[1].plot(hist['epoch'], hist['val_iou'],    label='val')
    axes[1].set_title('IoU'); axes[1].legend()
    plt.tight_layout(); plt.savefig(od / 'curves.png', dpi=100); plt.close()
    print(f'Experiment {name} complete. Best Val IoU = {best_iou:.4f}')
    return best_iou, ckpt

print('Helpers defined. Ready to train.')


## 10. Experiment A: 6-ch input, WeightedBCE+Dice
*Estimated time on T4: ~3-5 min/epoch, ~15-25 epochs with early stopping*

In [ ]:
exp_results = {}
iou_a, ckpt_a = run_experiment('A', EXPERIMENTS['A'], pos_weight_val)
exp_results['A'] = {'val_iou': iou_a, 'ckpt': str(ckpt_a)}


## 11. Experiment B: 9-ch [A,B,|A-B|] input, WeightedBCE+Dice

In [ ]:
iou_b, ckpt_b = run_experiment('B', EXPERIMENTS['B'], pos_weight_val)
exp_results['B'] = {'val_iou': iou_b, 'ckpt': str(ckpt_b)}


## 12. Experiment C: 9-ch [A,B,|A-B|] input, Focal+Dice

In [ ]:
iou_c, ckpt_c = run_experiment('C', EXPERIMENTS['C'], pos_weight_val)
exp_results['C'] = {'val_iou': iou_c, 'ckpt': str(ckpt_c)}


## 13. Compare experiments and select the best checkpoint

In [ ]:
print('=' * 60)
print('EXPERIMENT COMPARISON (Validation IoU)')
print('=' * 60)
print(f'{"Exp":<5} {"Val IoU":<10} Description')
print('-' * 60)
best_name = None; best_iou = -1.0
for nm, res in exp_results.items():
    star = ' <-- BEST' if res['val_iou'] == max(r['val_iou'] for r in exp_results.values()) else ''
    print(f'{nm:<5} {res["val_iou"]:<10.4f} {EXPERIMENTS[nm]["description"]}{star}')
    if res['val_iou'] > best_iou:
        best_iou = res['val_iou']; best_name = nm
print(f'\nWinner: Experiment {best_name}  (Val IoU = {best_iou:.4f})')
shutil.copy2(exp_results[best_name]['ckpt'], 'models/best_temporal_model.pth')
print('Copied -> models/best_temporal_model.pth')


## 14. Threshold sweep on VALIDATION set only

> The test set is **not touched** in this cell. Threshold is selected using validation IoU.

In [ ]:
ck = torch.load('models/best_temporal_model.pth', map_location=DEVICE, weights_only=False)
cfg_b = ck['config']
mb = TemporalUNet(in_channels=cfg_b['in_channels'], out_channels=1,
                  init_features=cfg_b['init_features'],
                  bottleneck_dropout=cfg_b['bottleneck_dropout']).to(DEVICE)
mb.load_state_dict(ck['model_state_dict']); mb.eval()
_, vl2, _ = get_temporal_dataloaders(DATASET_ROOT, BATCH_SIZE, NUM_WORKERS,
                                      diff_channels=cfg_b['diff_channels'])
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
print(f'Threshold sweep on VAL set (Exp {best_name}):')
print(f'{"Threshold":<12} {"Val IoU":<12} {"Val Dice":<12} {"Val Prec":<12} {"Val Rec"}')
best_t = 0.5; best_t_iou = -1.0; thresh_res = []
for t in thresholds:
    tp = fp = fn = tn = 0
    with torch.no_grad():
        for imgs, masks, _ in vl2:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            pr = (torch.sigmoid(mb(imgs)) > t).float()
            tp += (pr * masks).sum().item()
            fp += (pr * (1 - masks)).sum().item()
            fn += ((1 - pr) * masks).sum().item()
            tn += ((1 - pr) * (1 - masks)).sum().item()
    iou_t = iou_f(tp,fp,fn); dice_t = dice_f(tp,fp,fn)
    pr_t  = tp/(tp+fp) if (tp+fp)>0 else 1.0
    rc_t  = tp/(tp+fn) if (tp+fn)>0 else 1.0
    print(f'{t:<12.1f} {iou_t:<12.4f} {dice_t:<12.4f} {pr_t:<12.4f} {rc_t:.4f}')
    thresh_res.append({'threshold': t, 'iou': iou_t, 'dice': dice_t, 'precision': pr_t, 'recall': rc_t})
    if iou_t > best_t_iou: best_t_iou = iou_t; best_t = t
print(f'\nSelected threshold: {best_t}  (Val IoU = {best_t_iou:.4f})')


## 15. Final TEST set evaluation (one time only)

> **Scientific integrity:** The test set was never used during training or threshold selection.  
> It is evaluated **once** here at the threshold chosen on validation data.

In [ ]:
_, _, te = get_temporal_dataloaders(DATASET_ROOT, BATCH_SIZE, NUM_WORKERS,
                                    diff_channels=cfg_b['diff_channels'])
tp = fp = fn = tn = 0
mb.eval()
with torch.no_grad():
    for imgs, masks, _ in te:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pr = (torch.sigmoid(mb(imgs)) > best_t).float()
        tp += (pr * masks).sum().item()
        fp += (pr * (1 - masks)).sum().item()
        fn += ((1 - pr) * masks).sum().item()
        tn += ((1 - pr) * (1 - masks)).sum().item()
total = tp + fp + fn + tn
ti  = iou_f(tp, fp, fn); td  = dice_f(tp, fp, fn)
tpr = tp/(tp+fp) if (tp+fp)>0 else 1.0
trc = tp/(tp+fn) if (tp+fn)>0 else 1.0
tac = (tp+tn)/total if total>0 else 1.0
print('FINAL TEST RESULTS')
print(f'  Threshold  : {best_t}')
print(f'  IoU        : {ti:.4f}   (old baseline: 0.2983)')
print(f'  Dice (F1)  : {td:.4f}   (old baseline: 0.4595)')
print(f'  Precision  : {tpr:.4f}   (old baseline: 0.4131)')
print(f'  Recall     : {trc:.4f}   (old baseline: 0.5176)')
print(f'  Accuracy   : {tac:.4f}')
print(f'  TP={int(tp)}, FP={int(fp)}, FN={int(fn)}, TN={int(tn)}')
# Save results
out_dir = Path('outputs/temporal_v2'); out_dir.mkdir(parents=True, exist_ok=True)
results = {
    'best_experiment': best_name,
    'selected_threshold': best_t,
    'val_iou_at_threshold': best_t_iou,
    'threshold_sweep': thresh_res,
    'test': {'iou': ti, 'dice': td, 'precision': tpr, 'recall': trc, 'accuracy': tac,
             'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)},
    'baseline': {'iou': 0.2983, 'dice': 0.4595, 'precision': 0.4131, 'recall': 0.5176},
    'model_config': cfg_b,
    'experiment_val_ious': {k: v['val_iou'] for k, v in exp_results.items()},
}
with open(out_dir / 'final_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: outputs/temporal_v2/final_results.json')


## 16. Download the best model checkpoint

In [ ]:
from google.colab import files
import os
mp = 'models/best_temporal_model.pth'
size_mb = os.path.getsize(mp) / 1e6
print(f'Downloading {mp} ({size_mb:.1f} MB)...')
files.download(mp)
print('Download started. Check your browser downloads.')
print('Place the file at: models/best_temporal_model.pth in your local project.')


## 17. Download all results as a ZIP (optional)

In [ ]:
import zipfile
from google.colab import files
zpath = '/content/stage2_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path('outputs').rglob('*'):
        if p.is_file() and p.suffix in ['.json', '.txt', '.csv', '.png']:
            zf.write(p, p.relative_to('.'))
print(f'Created stage2_results.zip ({Path(zpath).stat().st_size/1e6:.1f} MB)')
files.download(zpath)


---
## After training: what to do with the downloaded files

### 1. Replace the local model checkpoint
Copy `best_temporal_model.pth` to your local project:
```
models\best_temporal_model.pth
```

### 2. Update `backend/main.py` if needed
The backend loads the checkpoint at startup:
```python
temporal_model = TemporalUNet(in_channels=6, ...)
```
If the winning experiment was **B or C** (9-channel), the backend `/predict-temporal`
endpoint must also compute `|A-B|` before model inference.
**Report back which experiment won**, and this update will be done for you.

### 3. Frontend requires no changes
The React UI sends raw Before + After images. Diff computation is server-side.

### Output files
| File | Description |
|------|-------------|
| `models/best_temporal_model.pth` | **Download this** |
| `outputs/experiments/exp_A/` | Loss/IoU curves + epoch CSV |
| `outputs/experiments/exp_B/` | Loss/IoU curves + epoch CSV |
| `outputs/experiments/exp_C/` | Loss/IoU curves + epoch CSV |
| `outputs/temporal_v2/final_results.json` | All metrics |
| `stage2_results.zip` | All logs/plots bundled |